# 01. 데이터 읽기 및 기본 확인

첫 단계에서는 원본 데이터를 변경하거나 전처리 결과를 저장하지 않는다. 데이터 경로, 파일 구성, 컬럼, 샘플 품질만 확인한다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root()
DATA_ROOT = (PROJECT_ROOT / '../../data/stock_data').resolve()
MODEL_ROOT = (PROJECT_ROOT / '../../model').resolve()
RESULTS_ROOT = (PROJECT_ROOT / '../../results').resolve()

for name, path in {
    'project': PROJECT_ROOT,
    'data': DATA_ROOT,
    'model': MODEL_ROOT,
    'results': RESULTS_ROOT,
}.items():
    print(f'{name:>7}: {path} | exists={path.exists()}')

assert DATA_ROOT.is_dir(), f'데이터 경로가 없습니다: {DATA_ROOT}'

project: /home/user/urbandatalab/YSLee/code/Detecting-entry-signals-for-short-term-trades-right-before-a-rapid-price-surge | exists=True
   data: /home/user/urbandatalab/YSLee/data/stock_data | exists=True
  model: /home/user/urbandatalab/YSLee/model | exists=True
results: /home/user/urbandatalab/YSLee/results | exists=True


## 파일과 세션 확인

In [3]:
files = sorted(DATA_ROOT.glob('raw/session_*/*_enriched.csv'))
assert files, f'enriched CSV가 없습니다: {DATA_ROOT}'

inventory = pd.DataFrame({
    'session': [path.parent.name for path in files],
    'file_name': [path.name for path in files],
    'source_file': [str(path) for path in files],
})

print(f'세션 수: {inventory.session.nunique():,}')
print(f'파일 수: {len(inventory):,}')
display(inventory.groupby('session').size().rename('file_count').to_frame())
display(inventory.head())

세션 수: 9
파일 수: 208


,file_count
session,
session_2026-07-07,14
session_2026-07-08,25
session_2026-07-09,33
session_2026-07-10,34
session_2026-07-13,21
session_2026-07-14,20
session_2026-07-15,23
session_2026-07-16,17
session_2026-07-17,21


,session,file_name,source_file
0,session_2026-07-07,alm_2026-07-07_1700-0800_kst_7ff6d430_enriched...,/home/user/urbandatalab/YSLee/data/stock_data/...
1,session_2026-07-07,bmnr_2026-07-07_1700-0800_kst_bfa81f10_enriche...,/home/user/urbandatalab/YSLee/data/stock_data/...
2,session_2026-07-07,buru_2026-07-07_1700-0800_kst_68d8051b_enriche...,/home/user/urbandatalab/YSLee/data/stock_data/...
3,session_2026-07-07,clro_2026-07-07_1700-0800_kst_a4e81094_enriche...,/home/user/urbandatalab/YSLee/data/stock_data/...
4,session_2026-07-07,dcx_2026-07-07_1700-0800_kst_02d88b1e_enriched...,/home/user/urbandatalab/YSLee/data/stock_data/...


## 전체 파일 스키마 확인

전체 파일은 헤더만 읽어 컬럼 구성이 같은지 확인한다.

In [4]:
REQUIRED_COLUMNS = ['symbol', 'timestamp_utc', 'open', 'high', 'low', 'close']
schema_rows = []

for path in files:
    columns = pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns.tolist()
    schema_rows.append({
        'session': path.parent.name,
        'file_name': path.name,
        'column_count': len(columns),
        'missing_required': sorted(set(REQUIRED_COLUMNS) - set(columns)),
        'schema': tuple(columns),
    })

schema_check = pd.DataFrame(schema_rows)
print(f'고유 스키마 수: {schema_check.schema.nunique()}')
print(f'필수 컬럼 누락 파일: {schema_check.missing_required.astype(bool).sum()}')
display(schema_check.groupby('column_count').size().rename('file_count').to_frame())
display(schema_check.head())

고유 스키마 수: 1
필수 컬럼 누락 파일: 0


,file_count
column_count,
55,208


,session,file_name,column_count,missing_required,schema
0,session_2026-07-07,alm_2026-07-07_1700-0800_kst_7ff6d430_enriched...,55,[],"(symbol, timestamp_kst, timestamp_utc, open, h..."
1,session_2026-07-07,bmnr_2026-07-07_1700-0800_kst_bfa81f10_enriche...,55,[],"(symbol, timestamp_kst, timestamp_utc, open, h..."
2,session_2026-07-07,buru_2026-07-07_1700-0800_kst_68d8051b_enriche...,55,[],"(symbol, timestamp_kst, timestamp_utc, open, h..."
3,session_2026-07-07,clro_2026-07-07_1700-0800_kst_a4e81094_enriche...,55,[],"(symbol, timestamp_kst, timestamp_utc, open, h..."
4,session_2026-07-07,dcx_2026-07-07_1700-0800_kst_02d88b1e_enriched...,55,[],"(symbol, timestamp_kst, timestamp_utc, open, h..."


## 샘플 파일 읽기

우선 첫 번째 파일 하나를 전체 로드한다. 종목 또는 날짜 선정 규칙은 아직 적용하지 않는다.

In [5]:
SAMPLE_PATH = files[0]
sample_raw = pd.read_csv(SAMPLE_PATH, encoding='utf-8-sig')

print(f'파일: {SAMPLE_PATH}')
print(f'크기: {sample_raw.shape}')
display(sample_raw.head())
display(sample_raw.dtypes.rename('dtype').to_frame())

파일: /home/user/urbandatalab/YSLee/data/stock_data/raw/session_2026-07-07/alm_2026-07-07_1700-0800_kst_7ff6d430_enriched.csv
크기: (446, 55)


,symbol,timestamp_kst,timestamp_utc,open,high,low,close,volume,trade_count,alpaca_vwap,notional_usd,volume_ma20,ma5,ma20,ma60,ma120,ema9,ema20,bb_mid20,bb_upper20,bb_lower20,bb_position_0_1,session_vwap,macd_12_26,macd_signal_9,macd_histogram,candle_color,return_pct,range_pct,quote_count,first_bid,first_ask,last_bid,last_ask,avg_bid,avg_ask,min_bid,max_bid,min_ask,max_ask,avg_spread_pct,min_spread_pct,max_spread_pct,avg_bid_size,avg_ask_size,bid_ask_imbalance,trade_tick_count,trade_volume_from_ticks,aggressive_buy_volume,aggressive_sell_volume,unknown_trade_volume,trade_notional_from_ticks,trade_strength,sell_pressure,unknown_trade_ratio
0,ALM,2026-07-07 17:00:00,2026-07-07 08:00:00,16.00,16.20,15.9847,16.20,4331,27,16.012325,70162.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.128233,NaN,NaN,NaN,up,NaN,1.329012,12.0,16.04,16.33,16.0,16.25,14.9925,16.31,12.0,16.04,16.25,16.33,8.723812,1.300712,30.568302,783.333333,775.0,0.502674,27,4331,0.0,100.0,4231.0,69370.8691,0.0,100.0,97.691064
1,ALM,2026-07-07 17:03:00,2026-07-07 08:03:00,16.01,16.08,16.0100,16.08,944,35,16.045000,15179.52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.115426,NaN,NaN,NaN,up,-0.740741,0.435323,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35,944,0.0,0.0,944.0,15139.0300,NaN,NaN,100.000000
2,ALM,2026-07-07 17:04:00,2026-07-07 08:04:00,16.00,16.20,15.9900,16.20,5086,79,16.046065,82393.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.122580,NaN,NaN,NaN,up,0.746269,1.296296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,79,5086,0.0,0.0,5086.0,81554.3600,NaN,NaN,100.000000
3,ALM,2026-07-07 17:05:00,2026-07-07 08:05:00,15.99,16.22,15.9900,16.22,4188,40,16.172957,67929.36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.128554,NaN,NaN,NaN,up,0.123457,1.418002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40,4188,0.0,0.0,4188.0,67694.2800,NaN,NaN,100.000000
4,ALM,2026-07-07 17:06:00,2026-07-07 08:06:00,16.06,16.20,16.0600,16.20,4156,8,16.196585,67327.20,NaN,16.18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.134060,NaN,NaN,NaN,up,-0.123305,0.864198,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8,4156,0.0,0.0,4156.0,67309.5800,NaN,NaN,100.000000


,dtype
symbol,object
timestamp_kst,object
timestamp_utc,object
open,float64
high,float64
low,float64
close,float64
volume,int64
trade_count,int64
alpaca_vwap,float64


## 샘플 기본 품질 확인

확인용 복사본에서 시간과 OHLC dtype만 정리한다. 원본 파일은 수정하지 않는다.

In [6]:
sample = sample_raw.copy()
sample['symbol'] = sample['symbol'].astype('string').str.strip().str.upper()
sample['timestamp_utc'] = pd.to_datetime(sample['timestamp_utc'], errors='coerce', utc=True)
for column in ['open', 'high', 'low', 'close']:
    sample[column] = pd.to_numeric(sample[column], errors='coerce')

required_missing = sample[REQUIRED_COLUMNS].isna().any(axis=1)
valid_ohlc = (
    sample[['open', 'high', 'low', 'close']].gt(0).all(axis=1)
    & sample['high'].ge(sample[['open', 'close']].max(axis=1))
    & sample['low'].le(sample[['open', 'close']].min(axis=1))
    & sample['high'].ge(sample['low'])
)
duplicate_key = sample.duplicated(['symbol', 'timestamp_utc'], keep=False)
ordered = sample.sort_values(['symbol', 'timestamp_utc'])
minute_delta = ordered.groupby('symbol')['timestamp_utc'].diff().dt.total_seconds().div(60)

quality_summary = pd.Series({
    'rows': len(sample),
    'symbols': sample.symbol.nunique(dropna=True),
    'start_utc': sample.timestamp_utc.min(),
    'end_utc': sample.timestamp_utc.max(),
    'required_missing_rows': int(required_missing.sum()),
    'invalid_ohlc_rows': int((~valid_ohlc).sum()),
    'duplicate_symbol_timestamp_rows': int(duplicate_key.sum()),
    'gaps_over_one_minute': int(minute_delta.gt(1).sum()),
}, name='value')
display(quality_summary.to_frame())
display(sample[REQUIRED_COLUMNS].head(10))

,value
rows,446
symbols,1
start_utc,2026-07-07 08:00:00+00:00
end_utc,2026-07-07 22:51:00+00:00
required_missing_rows,0
invalid_ohlc_rows,0
duplicate_symbol_timestamp_rows,0
gaps_over_one_minute,37


,symbol,timestamp_utc,open,high,low,close
0,ALM,2026-07-07 08:00:00+00:00,16.0000,16.2000,15.9847,16.2000
1,ALM,2026-07-07 08:03:00+00:00,16.0100,16.0800,16.0100,16.0800
2,ALM,2026-07-07 08:04:00+00:00,16.0000,16.2000,15.9900,16.2000
3,ALM,2026-07-07 08:05:00+00:00,15.9900,16.2200,15.9900,16.2200
4,ALM,2026-07-07 08:06:00+00:00,16.0600,16.2000,16.0600,16.2000
5,ALM,2026-07-07 08:30:00+00:00,16.2400,16.2400,16.2400,16.2400
6,ALM,2026-07-07 08:48:00+00:00,16.2400,16.2400,16.2400,16.2400
7,ALM,2026-07-07 09:07:00+00:00,16.2400,16.2400,16.2400,16.2400
8,ALM,2026-07-07 09:34:00+00:00,16.2268,16.2268,16.2268,16.2268
9,ALM,2026-07-07 09:42:00+00:00,16.2068,16.2068,16.2068,16.2068


## 현재 단계 결론

이 노트북은 데이터를 읽고 구조를 확인하는 용도다. 다음 단계에서 전체 파일 품질 집계와 실제 정제 기준을 별도로 결정한다. 현재는 gap 제거, 결측 보간, feature 생성, 라벨 생성, 데이터 저장을 수행하지 않는다.